In [1]:
import pandas as pd
from FinMind.data import DataLoader
import datetime
import time

def check_cost_concentration(api, ticker, lookback_days=120, threshold=0.15):
    try:
        # 1. Fetch data with a buffer for non-trading days
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=lookback_days + 60) 
        
        # 2. Call FinMind API
        df = api.taiwan_stock_daily(
            stock_id=ticker,
            start_date=start_date.strftime('%Y-%m-%d'),
            end_date=end_date.strftime('%Y-%m-%d')
        )
        
        # Check if data exists and is long enough
        if df is None or df.empty or len(df) < lookback_days:
            return None
            
        # 3. Keep only the exact lookback window
        df = df.tail(lookback_days).copy()
        current_price = df['close'].iloc[-1]
        
        # 4. Calculate Typical Price 
        # FinMind uses 'max', 'min', 'close', and 'Trading_Volume'
        df['Typical_Price'] = ((df['max'] + df['min'] + df['close']) / 3).round(2)
        
        # 5. Build Volume Profile
        profile = df.groupby('Typical_Price')['Trading_Volume'].sum().reset_index()
        profile = profile.sort_values(by='Typical_Price')
        profile['Cum_Volume'] = profile['Trading_Volume'].cumsum()
        
        total_volume = profile['Trading_Volume'].sum()
        if total_volume == 0:
            return None
            
        # 6. Find 15% and 85% volume percentiles (The middle 70% bulk)
        lower_bound_vol = total_volume * 0.15
        upper_bound_vol = total_volume * 0.85
        
        lower_price = profile.loc[profile['Cum_Volume'] >= lower_bound_vol, 'Typical_Price'].iloc[0]
        upper_price = profile.loc[profile['Cum_Volume'] >= upper_bound_vol, 'Typical_Price'].iloc[0]
        
        # 7. Calculate concentration ratio
        concentration = (upper_price - lower_price) / current_price
        
        if concentration < threshold:
            return {
                'Ticker': ticker,
                'Current Price': current_price,
                'Concentration %': round(concentration * 100, 2),
                'Lower Cost': lower_price,
                'Upper Cost': upper_price
            }
        return None
        
    except Exception as e:
        print(f"Error evaluating {ticker}: {e}")
        return None

def run_daily_pipeline():
    # Initialize FinMind DataLoader
    api = DataLoader()
    
    # Optional: Login to increase your API limits!
    # api.login_by_token(api_token='YOUR_FINMIND_TOKEN')
    
    # Example Tickers (No .TW/.TWO suffixes needed for FinMind)
    tw_tickers = ['2330', '2317', '2454', '3231', '2603', '3529'] 
    
    results = []
    print(f"Running Cost Concentration Scan via FinMind for {datetime.date.today()}...")
    
    for ticker in tw_tickers:
        res = check_cost_concentration(api, ticker)
        if res:
            results.append(res)
            
        # VERY IMPORTANT: Rate limiting
        time.sleep(1) 
            
    if results:
        results_df = pd.DataFrame(results)
        print("\n--- Stocks passing the 70% Cost Concentration (<15%) Filter ---")
        print(results_df.to_markdown(index=False))
    else:
        print("\nNo stocks met the criteria today.")

if __name__ == "__main__":
    run_daily_pipeline()

2026-03-20 16:23:42.829 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 2330


Running Cost Concentration Scan via FinMind for 2026-03-20...


2026-03-20 16:23:45.003 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-03-20 16:23:46.095 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 2454
2026-03-20 16:23:47.182 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 3231
2026-03-20 16:23:48.274 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 2603
2026-03-20 16:23:49.363 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 3529



No stocks met the criteria today.
